# Model C: a small LLM (Qwen2.5-1.5B-Instruct), zero-shot and LoRA

The LLM is asked: *two headlines for the same story were A/B tested; which got more clicks? Answer A or B.* Its answer is read from the scores it gives to the tokens "A" and "B", turned into a probability. Every pair is asked in both orders and the two answers are averaged, so the model cannot win by always saying "A".

1. **Zero-shot:** no training, on all 3,860 pairs. Tests what the LLM already "knows" about clickability.
2. **LoRA:** small trainable adapters (lecture 11) on the attention layers, trained on the same 5 folds as DeBERTa, 1 epoch, last epoch used. Directly comparable to DeBERTa's 0.752.

Each result is saved as soon as it finishes, and finished parts are skipped on a rerun.

**Runtime: roughly 2 to 3 hours on Kaggle** (the model runs in full 32-bit precision for stability). Settings: Accelerator GPU T4, Internet **On**, input dataset `good-news-cv`. Then Save Version, Save & Run All (Commit).

## 1. Check the GPU

In [1]:
!nvidia-smi

Thu Nov 19 21:35:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install packages
`peft` provides LoRA.

In [2]:
!pip install -q peft
!pip uninstall -y -q torchao

## 3. Storage and data

In [3]:
import glob, os, shutil

if os.path.exists("/kaggle"):
    OUT_DIR = "/kaggle/working"
    found = glob.glob("/kaggle/input/**/pairs_cv.csv", recursive=True)
    assert found, "Add your good-news-cv dataset (Add Input, right panel)"
    shutil.copy(found[0], f"{OUT_DIR}/pairs_cv.csv")
else:
    from google.colab import drive, files
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/good-news-nlp-llm"
    os.makedirs(OUT_DIR, exist_ok=True)
    if not os.path.exists(f"{OUT_DIR}/pairs_cv.csv"):
        uploaded = files.upload()
        name = next(n for n in uploaded if n.startswith("pairs_cv"))
        shutil.move(name, f"{OUT_DIR}/pairs_cv.csv")

print("Saving to:", OUT_DIR)
print("Already finished:", sorted(os.path.basename(f) for f in glob.glob(f"{OUT_DIR}/*llm*.csv")))

Saving to: /kaggle/working
Already finished: []


## 4. Settings

In [4]:
import glob, os, random, time
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

CONFIG = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",  # open, not gated, fits a T4
    "max_length": 160,
    "eval_batch_size": 32,
    "train_batch_size": 8,
    "grad_accumulation": 2,          # effective batch 16
    "learning_rate": 1e-4,
    "epochs": 1,
    "warmup_ratio": 0.1,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "seed": 42,
    "n_folds": 5,
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int) -> None:
    """Fix every source of randomness so a rerun gives the same result."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("Device:", DEVICE)

Device: cuda


## 5. Data, tokenizer and the prompt
Prints one example prompt so you can see exactly what the model is asked.

In [5]:
pairs = pd.read_csv(f"{OUT_DIR}/pairs_cv.csv")
print(f"Pairs: {len(pairs)} in {pairs['story_group'].nunique()} stories, {pairs['fold'].nunique()} folds")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.padding_side = "left"   # so the last position is always where the answer goes
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM = "You are an expert in online news and audience behaviour."
QUESTION = ("Two headlines for the same story were A/B tested on a news website. "
            "Which headline got more clicks?\n\nA: {a}\nB: {b}\n\n"
            "Answer with a single letter: A or B.")


def build_prompt(a: str, b: str) -> str:
    """Chat-formatted prompt that ends exactly where the model writes its answer."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": QUESTION.format(a=a, b=b)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# The model's answer is read from the scores it gives to the tokens "A" and "B".
ID_A = tokenizer.encode("A", add_special_tokens=False)[0]
ID_B = tokenizer.encode("B", add_special_tokens=False)[0]
print("Answer token ids:", ID_A, ID_B)
print("\nExample prompt:\n" + build_prompt("Headline one", "Headline two"))

Pairs: 3860 in 1000 stories, 5 folds


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Answer token ids: 32 33

Example prompt:
<|im_start|>system
You are an expert in online news and audience behaviour.<|im_end|>
<|im_start|>user
Two headlines for the same story were A/B tested on a news website. Which headline got more clicks?

A: Headline one
B: Headline two

Answer with a single letter: A or B.<|im_end|>
<|im_start|>assistant



## 6. Functions

In [6]:
def load_model():
    """Load the LLM in float32 (half precision gave NaN for DeBERTa, so we avoid it here)."""
    try:
        model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], dtype=torch.float32)
    except TypeError:  # older transformers versions
        model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], torch_dtype=torch.float32)
    return model.to(DEVICE)


def ab_logits(model, a_list, b_list) -> torch.Tensor:
    """Return [score for 'A', score for 'B'] at the answer position, one row per pair."""
    prompts = [build_prompt(a, b) for a, b in zip(a_list, b_list)]
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                    max_length=CONFIG["max_length"]).to(DEVICE)
    last = model(**enc).logits[:, -1, :]
    return last[:, [ID_A, ID_B]].float()


@torch.no_grad()
def predict_proba(model, df: pd.DataFrame) -> np.ndarray:
    """P(headline_a wins), averaged over both orders so position cannot bias it."""
    model.eval()
    a, b = list(df["headline_a"]), list(df["headline_b"])
    bs = CONFIG["eval_batch_size"]

    def run(first, second):
        out = []
        for i in range(0, len(first), bs):
            out.append(torch.softmax(ab_logits(model, first[i:i + bs], second[i:i + bs]), dim=-1)[:, 0].cpu())
        return torch.cat(out).numpy()   # probability that the FIRST headline wins

    return (run(a, b) + (1 - run(b, a))) / 2


def save_predictions(df: pd.DataFrame, probs: np.ndarray, path: str) -> float:
    """Write predictions in the same format as the other models; return accuracy."""
    out = df[["test_id", "story_group", "fold", "headline_a", "headline_b", "label"]].copy()
    out["prob_a_wins"] = probs
    out["prediction"] = (probs > 0.5).astype(int)
    out["correct"] = out["prediction"] == out["label"]
    out.to_csv(path, index=False)
    return out["correct"].mean()


def train_lora(train_df: pd.DataFrame):
    """Fresh LLM + LoRA adapters, trained to put a higher score on the winning letter."""
    set_seed(CONFIG["seed"])
    model = load_model()
    lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
                      lora_dropout=CONFIG["lora_dropout"], task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    model = get_peft_model(model, lora)

    # Swap augmentation: every pair is also shown reversed, with the answer flipped.
    a = list(train_df["headline_a"]) + list(train_df["headline_b"])
    b = list(train_df["headline_b"]) + list(train_df["headline_a"])
    y = list(1 - train_df["label"]) + list(train_df["label"])   # 0 = "A" wins, 1 = "B" wins
    order = np.random.default_rng(CONFIG["seed"]).permutation(len(y))

    bs, accum = CONFIG["train_batch_size"], CONFIG["grad_accumulation"]
    steps_per_epoch = int(np.ceil(len(y) / bs / accum))
    total = steps_per_epoch * CONFIG["epochs"]
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=CONFIG["learning_rate"])
    scheduler = get_linear_schedule_with_warmup(optimizer, int(CONFIG["warmup_ratio"] * total), total)

    model.train()
    for epoch in range(CONFIG["epochs"]):
        running, n_batches = 0.0, 0
        for i in range(0, len(y), bs):
            idx = order[i:i + bs]
            logits = ab_logits(model, [a[j] for j in idx], [b[j] for j in idx])
            target = torch.tensor([y[j] for j in idx], device=DEVICE)
            loss = torch.nn.functional.cross_entropy(logits, target) / accum
            if torch.isnan(loss):
                raise RuntimeError("NaN loss")
            loss.backward()
            n_batches += 1
            running += loss.item() * accum
            if n_batches % accum == 0 or i + bs >= len(y):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            if n_batches % 100 == 0:
                print(f"    step {n_batches}: mean loss {running / n_batches:.4f}")
        print(f"  epoch {epoch + 1}: mean train loss {running / n_batches:.4f}")
    return model

## 7. Zero-shot on all pairs
No training. About 5 to 10 minutes.

In [7]:
zs_path = f"{OUT_DIR}/cv_predictions_llm_zeroshot.csv"
if os.path.exists(zs_path):
    print("Zero-shot predictions already saved, skipping")
else:
    start = time.time()
    model = load_model()
    probs = predict_proba(model, pairs)
    acc = save_predictions(pairs, probs, zs_path)
    print(f"Zero-shot accuracy on all {len(pairs)} pairs: {acc:.3f}  ({(time.time() - start) / 60:.1f} min)")
    del model
    torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Zero-shot accuracy on all 3860 pairs: 0.574  (12.8 min)


## 8. LoRA, 5 folds
A fresh model per fold, trained on the other 4 folds. Prints the loss every 100 steps. The long part.

In [8]:
start = time.time()
for k in range(CONFIG["n_folds"]):
    pred_path = f"{OUT_DIR}/llm_lora_cv_fold{k}.csv"
    if os.path.exists(pred_path):
        print(f"Fold {k}: already saved, skipping")
        continue
    train_df = pairs[pairs["fold"] != k].reset_index(drop=True)
    held_df = pairs[pairs["fold"] == k].reset_index(drop=True)
    print(f"Fold {k}: train {len(train_df)} pairs, predict {len(held_df)} pairs")
    model = train_lora(train_df)
    acc = save_predictions(held_df, predict_proba(model, held_df), pred_path)
    print(f"  fold {k} accuracy {acc:.3f}  (saved, {(time.time() - start) / 60:.1f} min this session)\n")
    del model
    torch.cuda.empty_cache()
print("All folds done.")

Fold 0: train 3089 pairs, predict 771 pairs


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    step 100: mean loss 0.6832
    step 200: mean loss 0.6748
    step 300: mean loss 0.6440
    step 400: mean loss 0.6019
    step 500: mean loss 0.5713
    step 600: mean loss 0.5495
    step 700: mean loss 0.5347
  epoch 1: mean train loss 0.5229
  fold 0 accuracy 0.757  (saved, 24.9 min this session)

Fold 1: train 3088 pairs, predict 772 pairs


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    step 100: mean loss 0.6995
    step 200: mean loss 0.6853
    step 300: mean loss 0.6597
    step 400: mean loss 0.6394
    step 500: mean loss 0.6094
    step 600: mean loss 0.5868
    step 700: mean loss 0.5623
  epoch 1: mean train loss 0.5482
  fold 1 accuracy 0.760  (saved, 49.9 min this session)

Fold 2: train 3089 pairs, predict 771 pairs


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    step 100: mean loss 0.6957
    step 200: mean loss 0.6923
    step 300: mean loss 0.6616
    step 400: mean loss 0.6387
    step 500: mean loss 0.6103
    step 600: mean loss 0.5817
    step 700: mean loss 0.5602
  epoch 1: mean train loss 0.5412
  fold 2 accuracy 0.741  (saved, 74.8 min this session)

Fold 3: train 3086 pairs, predict 774 pairs


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    step 100: mean loss 0.7054
    step 200: mean loss 0.6807
    step 300: mean loss 0.6444
    step 400: mean loss 0.6035
    step 500: mean loss 0.5843
    step 600: mean loss 0.5645
    step 700: mean loss 0.5356
  epoch 1: mean train loss 0.5272
  fold 3 accuracy 0.765  (saved, 100.0 min this session)

Fold 4: train 3088 pairs, predict 772 pairs


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    step 100: mean loss 0.6877
    step 200: mean loss 0.6638
    step 300: mean loss 0.6384
    step 400: mean loss 0.6048
    step 500: mean loss 0.5823
    step 600: mean loss 0.5600
    step 700: mean loss 0.5415
  epoch 1: mean train loss 0.5314
  fold 4 accuracy 0.744  (saved, 125.0 min this session)

All folds done.


## 9. Summary
Download `cv_predictions_llm_zeroshot.csv` and `cv_predictions_llm_lora.csv` from the Output tab into `data/processed`.

In [9]:
folds = [pd.read_csv(f) for f in sorted(glob.glob(f"{OUT_DIR}/llm_lora_cv_fold*.csv"))]
assert len(folds) == CONFIG["n_folds"], f"Only {len(folds)} folds finished: run the LoRA cell again"
cv = pd.concat(folds, ignore_index=True)
cv.to_csv(f"{OUT_DIR}/cv_predictions_llm_lora.csv", index=False)
zs = pd.read_csv(f"{OUT_DIR}/cv_predictions_llm_zeroshot.csv")
print(f"LLM zero-shot accuracy:       {zs['correct'].mean():.3f}")
print(f"LLM LoRA cross-validated acc: {cv['correct'].mean():.3f}")
print(cv.groupby("fold")["correct"].mean().round(3).to_string())

LLM zero-shot accuracy:       0.574
LLM LoRA cross-validated acc: 0.753
fold
0    0.757
1    0.760
2    0.741
3    0.765
4    0.744
